In [ ]:

# Install required libraries
!pip install langchain transformers datasets chromadb
!pip install -U langchain-community
!pip install pypdf
!pip install sentence-transformers
!pip install chromadb
!pip install protobuf>=5.0,<6.0


  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.5
    Uninstalling protobuf-4.25.5:
      Successfully uninstalled protobuf-4.25.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.2 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.1 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.16.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 6.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.19
    Uninstalling langchain-core-0.3.19:
      Successfully uninstalled langchain-core-0.3.19
  Attempting uninstall: langchain
    Found existi

In [ ]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from transformers import pipeline
import pypdf
from langchain.schema import Document
import torch
from transformers import pipeline
from langchain.prompts import ChatPromptTemplate

In [130]:
def chunk_text_at_newline(text, max_chunk_size=25):
    """
    Splits text into chunks, ensuring boundaries are at \n and each chunk is within max_chunk_size.
    
    Args:
        text (str): The text to split.
        max_chunk_size (int): The maximum character size for each chunk.

    Returns:
        List[str]: A list of chunks.
    """
    # Split text into sections by newline
    sections = text.split('\n')
    
    chunks = []
    current_chunk = []
    current_size = 0
    
    for section in sections:
        section_size = len(section)
        
        # If adding this section exceeds the max_chunk_size, finalize the current chunk
        if current_size + section_size > max_chunk_size:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_size = 0
        
        # Add the current section to the chunk
        current_chunk.append(section)
        current_size += section_size + 1  # Include newline character in size
    
    # Add the last chunk if there's any remaining content
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    
    return chunks

text = "\n".join([page.page_content for page in pages])


chunks = chunk_text_at_newline(text, max_chunk_size=1)

# Assuming `chunks` is a list of strings
documents = [Document(page_content=chunk) for chunk in chunks]


for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:{chunk}\n{'-'*30}")


Chunk 1:
------------------------------
Chunk 2:product/material produced 
------------------------------
Chunk 3:manufacturer 
------------------------------
Chunk 4:distributed by 
------------------------------
Chunk 5:industry 
------------------------------
Chunk 6:position held 
------------------------------
Chunk 7:original broadcaster 
------------------------------
Chunk 8:owned by 
------------------------------
Chunk 9:founded by 
------------------------------
Chunk 10:distribution format 
------------------------------
Chunk 11:headquarters location 
------------------------------
Chunk 12:stock exchange 
------------------------------
Chunk 13:currency 
------------------------------
Chunk 14:parent organization 
------------------------------
Chunk 15:chief executive officer 
------------------------------
Chunk 16:director/manager 
------------------------------
Chunk 17:owner of 
------------------------------
Chunk 18:operator 
------------------------------
Chunk 19

In [122]:
# ----- Data Indexing Process -----
pdf_file_name = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/01_Input/01_FinRED_data/all_relations.pdf'

# Load your PDF document
loader = PyPDFLoader(pdf_file_name)
pages = loader.load()

# Split the document into smaller chunks (chunk_size=500)
#text_splitter = RecursiveCharacterTextSplitter(chunk_size=32, chunk_overlap=20)
#chunks = text_splitter.split_documents(pages)

text = "\n".join([page.page_content for page in pages])
chunks = chunk_text_at_newline(text, max_chunk_size=5)
documents = [Document(page_content=chunk) for chunk in chunks]


# Use Hugging Face `distilbert-base-nli-mean-tokens` for embedding
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distilbert-base-nli-mean-tokens")

# Embed the chunks as vectors and load them into the database
CHROMA_PATH = "ChromaDB"  # Path for Chroma vector database
db_chroma = Chroma.from_documents(documents, embeddings, persist_directory=CHROMA_PATH)


# ----- Retrieval and Generation Process -----
device = 0 if torch.cuda.is_available() else -1  # Use GPU if available, otherwise CPU



# Define prompt template with examples
PROMPT_TEMPLATE = """
The sentence "{question}" is a news line from a financial context.
Using the {context} as reference, determine the applicable relationship mentioned in the sentence.
Only provide the name of the relationship in one word or a very short phrase.
Do not include any additional information or explanations.
"""



## Trying google's large model

In [123]:

generator = pipeline('text2text-generation', model='google/flan-t5-large')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


### Lets try it out

In [124]:
query =  "Iphone is manufactured by compaby called Apple Inc."

# Retrieve context - top n most relevant chunks
docs_chroma = db_chroma.similarity_search_with_score(query, k=1)
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

prompt = prompt_template.format(context=context_text, question=query)

response = generator(prompt, max_new_tokens=100)

answer = response[0]['generated_text'].strip().split('\n')[0]
print("For the query: ", query)
print("Identified relationship is:", answer)

For the query:  Iphone is manufactured by compaby called Apple Inc.
Identified relationship is: manufacturer


In [125]:
query = "Mukesh Ambani is Chief Executive Officer of company called Reliance Inc."

# Retrieve context - top n most relevant chunks
docs_chroma = db_chroma.similarity_search_with_score(query, k=1)
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

prompt = prompt_template.format(context=context_text, question=query)

response = generator(prompt, max_new_tokens=100)

answer = response[0]['generated_text'].strip().split('\n')[0]
print("For the query: ", query)
print("Identified relationship is:", answer)

For the query:  Mukesh Ambani is Chief Executive Officer of company called Reliance Inc.
Identified relationship is: Chief Executive Officer


In [126]:
query = "Tesla Inc is owned by Elon Musk"

# Retrieve context - top n most relevant chunks
docs_chroma = db_chroma.similarity_search_with_score(query, k=1)
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

prompt = prompt_template.format(context=context_text, question=query)

response = generator(prompt, max_new_tokens=100)

answer = response[0]['generated_text'].strip().split('\n')[0]
print("For the query: ", query)
print("Identified relationship is:", answer)

For the query:  Tesla Inc is owned by Elon Musk
Identified relationship is: owner


In [127]:
query = "Inventories is distributed by dealers"

# Retrieve context - top n most relevant chunks
docs_chroma = db_chroma.similarity_search_with_score(query, k=1)
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

prompt = prompt_template.format(context=context_text, question=query)

response = generator(prompt, max_new_tokens=100)

answer = response[0]['generated_text'].strip().split('\n')[0]
print("For the query: ", query)
print("Identified relationship is:", answer)

For the query:  Inventories is distributed by dealers
Identified relationship is: distributor


In [131]:
query = "Many companies have headquarters in Mumbai"

# Retrieve context - top n most relevant chunks
docs_chroma = db_chroma.similarity_search_with_score(query, k=1)
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

prompt = prompt_template.format(context=context_text, question=query)

response = generator(prompt, max_new_tokens=100)

answer = response[0]['generated_text'].strip().split('\n')[0]
print("For the query: ", query)
print("Identified relationship is:", answer)

For the query:  Many companies have headquarters in Mumbai
Identified relationship is: headquarters
